In [1]:
import pickle
import numpy as np
import tensorflow as tf
import hls4ml

2026-08-30 19:58:13.595731: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-30 19:58:13.596842: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 19:58:13.615259: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 19:58:13.615778: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-30 19:58:14.068809: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [ ]:
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
MODEL_PATH = "t1_model.h5"

#N_EVAL_SAMPLES = 2000


def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]
 
 
def accuracy_from_probs(y_true_onehot, y_pred_probs):
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    return float(np.mean(y_true == y_pred))

In [22]:
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)
X_eval = X_test
y_eval = y_test

print("Formato do conjunto usado na comparação:", X_eval.shape, y_eval.shape)
print(f"\nCarregando modelo de {MODEL_PATH} ...")
keras_model = tf.keras.models.load_model(MODEL_PATH)
keras_model.summary()

Lendo ./dataset/sat6_test.pkl ...
Formato do conjunto usado na comparação: (81000, 32, 32, 4) (81000, 6)

Carregando modelo de t1_model.h5 ...
Model: "t1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_32x32x4 (InputLayer)  [(None, 32, 32, 4)]       0         
                                                                 
 conv1_3x3_s2 (Conv2D)       (None, 16, 16, 32)        1184      
                                                                 
 conv1_relu (Activation)     (None, 16, 16, 32)        0         
                                                                 
 conv2_1x1_s1 (Conv2D)       (None, 16, 16, 8)         264       
                                                                 
 conv2_relu (Activation)     (None, 16, 16, 8)         0         
                                                                 
 flatten (Flatten)           (None, 2048)            

In [ ]:
OUTPUT_DIR = "hls4ml_prj_t1"
FPGA_PART = "xc7z020clg400-1"
CLOCK_PERIOD_NS = 10
IO_TYPE = "io_stream"
BACKEND = "Vitis"

hls_config = hls4ml.utils.config_from_keras_model(
    keras_model,
    granularity="model",
)
hls_config["Model"]["Precision"] = "ap_fixed<24,10,AP_RND,AP_SAT>"
hls_config["Model"]["ReuseFactor"] = 1

print("\nConfiguração hls4ml gerada (ajustada):")
print(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    keras_model,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    part=FPGA_PART,
    clock_period=CLOCK_PERIOD_NS,
    io_type=IO_TYPE,
    backend=BACKEND,
)

hls_model.compile()


Configuração hls4ml gerada (ajustada):
{'Model': {'Precision': 'ap_fixed<24,10,AP_RND,AP_SAT>', 'ReuseFactor': 1, 'Strategy': 'Latency', 'BramFactor': 1000000000, 'TraceOutput': False}}


In [27]:
print("\nRodando predição com o modelo Keras...")
y_pred_keras = keras_model.predict(X_eval)

print("Rodando predição com o modelo hls4ml (csim)...")
y_pred_hls = hls_model.predict(X_eval)


Rodando predição com o modelo Keras...
2532/2532 [==============================] - 1s 309us/step
Rodando predição com o modelo hls4ml (csim)...


In [28]:
acc_keras = accuracy_from_probs(y_eval, y_pred_keras)
acc_hls = accuracy_from_probs(y_eval, y_pred_hls)

diff = np.abs(y_pred_keras - y_pred_hls)

print("\n===== Resultado da comparação =====")
print(f"Acurácia Keras (float)       : {acc_keras*100:.2f}%")
print(f"Acurácia hls4ml (csim)       : {acc_hls*100:.2f}%")
print(f"Diferença de acurácia        : {abs(acc_keras - acc_hls)*100:.2f} pontos percentuais")
print(f"Diferença máxima nas probs   : {diff.max():.6f}")
print(f"Diferença média nas probs    : {diff.mean():.6f}")

from sklearn.metrics import accuracy_score

print("Keras Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_pred_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_pred_hls, axis=1))))



===== Resultado da comparação =====
Acurácia Keras (float)       : 98.73%
Acurácia hls4ml (csim)       : 98.63%
Diferença de acurácia        : 0.10 pontos percentuais
Diferença máxima nas probs   : 0.186391
Diferença média nas probs    : 0.001585
Keras Accuracy: 0.9872962962962963
hls4ml Accuracy: 0.9862592592592593


In [32]:
hls_model.build(csim=True, synth=True)


****** vitis-run v2024.2 (64-bit)
  **** SW Build 5239630 on 2024-11-10-11:19:46
  **** Start of session at: Sun Aug 30 20:22:50 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2024.2 5238294
Sourcing Tcl script '/home/fauri/Documentos/hls4ml/sat_6_tensorflow/hls4ml_prj_t1/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Creating and opening project '/home/fauri/Documentos/hls4ml/sat_6_tensorflow/hls4ml_prj_t1/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 200-1510] Running: add_files -tb myproject_test.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding test bench file 'myproject_test.cpp' to the project
INFO: [HLS 200-1510] Running: add_files -

KeyboardInterrupt: 